# All Steps of Piepline

In [2]:
import sys
import os
import pandas as pd

# Add the project root directory to sys.path
project_root = os.path.abspath("../../")  # Adjust the relative path as needed
if project_root not in sys.path:
    sys.path.append(project_root)

## Bulk download of CRS data

In [ ]:
# Import the necessary functions from src
from src.download_crs import get_full_crs_parquet_url, download_crs_parquet

# Get the full CRS parquet URL
file_url = get_full_crs_parquet_url()

# Download the CRS data
crs = download_crs_parquet(file_url)

In [ ]:
# Save raw CRS to feather 
crs.to_feather("../../data/raw/crs_raw.feather")

## A - Title Pattern Matching 

###  Keyword Detection

In [14]:
from src.text_processing import normalize_str, detect_language, lemmatize_str, lemmatize_batch, detect_keywords, detect_acronyms, process_keywords

#### Load keywords

In [11]:
import pandas as pd
# Load keywords from Excel file
keywords_file_path = "../../data/keywords/review/keyword_review_modernization.xlsx"
stat_keywords = pd.read_excel(keywords_file_path, sheet_name="statistics")
stat_acronyms = pd.read_excel(keywords_file_path, sheet_name="statistics acronyms")
stat_blacklist = pd.read_excel(keywords_file_path, sheet_name="stat blacklist")
gender_keywords = pd.read_excel(keywords_file_path, sheet_name="gender")
gender_acronyms = pd.read_excel(keywords_file_path, sheet_name="gender acronyms")

# Drop id column for all keywords with loop
for df in [stat_keywords, stat_acronyms, stat_blacklist, 
           gender_keywords, gender_acronyms]:
    if 'id' in df.columns:
        df.drop(columns=['id'], inplace=True)
    
    print(f"{df.columns.tolist()}")


# Process keywords
stat_keywords = process_keywords(stat_keywords, remove_stopwords=True)
gender_keywords = process_keywords(gender_keywords, remove_stopwords=True)
stat_blacklist = process_keywords(stat_blacklist, remove_stopwords=True)

# Lowercase all strings in the acronyms dataframes
stat_acronyms = stat_acronyms.map(lambda x: x.lower() if isinstance(x, str) else x)
gender_acronyms = gender_acronyms.map(lambda x: x.lower() if isinstance(x, str) else x)

['en', 'fr', 'es', 'de']
['en', 'fr', 'es', 'de']
['en', 'fr', 'es', 'de']
['en', 'fr', 'es', 'de']
['en', 'fr', 'es', 'de']


#### Load CRS data

In [4]:
# Load CRS data from feather file
crs = pd.read_feather("../../data/raw/crs_raw.feather")

# Apply process row function to the first 100 rows of the dataframe
#crs = crs_raw.head(100000).copy()
#crs = crs.copy()

#del crs_raw

In [5]:
# Reduce crs to only the columns we need
crs = crs[['project_title', 'short_description', 'long_description']]

# Only keep rows with project_title that are unique and keep the first occurrence
crs = crs.drop_duplicates(subset=['project_title'], keep='first')

# Remove row with NaN values in the project_title column (drop_dupicates keeps one None value row)
crs = crs[crs['project_title'].notna()]

#### Detect keywords

In [6]:
# Process titles 
crs['normalized_title'] = crs['project_title'].apply(normalize_str)
crs['language'] = crs['normalized_title'].apply(detect_language)

In [7]:
# Lemmatize in batches (for entire CRS dataset ~23min on Intel i7 Gen 11)
for lang in crs['language'].unique():
    # Filter the DataFrame for the current language
    lang_df = crs[crs['language'] == lang]

    if 'lemmatized_title' not in crs.columns:
        crs['lemmatized_title'] = None  # Initialize the column if it doesn't exist
    crs['lemmatized_title'] = crs['lemmatized_title'].astype(object)
    
    # Process the batch and update the original DataFrame
    crs.loc[lang_df.index, 'lemmatized_title'] = lemmatize_batch(lang_df['normalized_title'].tolist(), lang, batch_size=1000, remove_stopwords=True)

    # Lowercase lemmatized titles in case spacy lemmatizes to uppercase
    crs['lemmatized_title'] = crs['lemmatized_title'].str.lower()

In [ ]:
# Save afer lemmatization 
#crs.to_feather("../../data/processed/crs_lemmatized_titles.feather")
#crs.to_feather("../../data/processed/crs_lemmatized_titles_wo_stopwords.feather")

In [9]:
# Load the lemmatized CRS data from feather file
#crs = pd.read_feather("../../data/processed/crs_lemmatized_titles.feather")
crs = pd.read_feather("../../data/processed/crs_lemmatized_titles_wo_stopwords.feather")

In [16]:
# Detect keywords in the lemmatized title
crs['stat_keywords'] = crs.apply(lambda row: detect_keywords(row['lemmatized_title'], row['language'], stat_keywords), axis=1)
crs['stat_blacklist'] = crs.apply(lambda row: detect_keywords(row['lemmatized_title'], row['language'], stat_blacklist), axis=1)
crs['gen_keywords'] = crs.apply(lambda row: detect_keywords(row['lemmatized_title'], row['language'], gender_keywords), axis=1)

# Detect acronyms in the normalized title
crs['stat_acronyms'] = crs.apply(lambda row: detect_acronyms(row['normalized_title'], row['language'], stat_acronyms), axis=1)
crs['gen_acronyms'] = crs.apply(lambda row: detect_acronyms(row['normalized_title'], row['language'], gender_acronyms), axis=1)

In [17]:
# Save result
#crs.to_feather("../../data/processed/crs_titles_matched.feather")
crs.to_feather("../../data/processed/crs_titles_matched_wo_stopwords.feather")


In [19]:
# Reduce crs to only rows with keywords and acronyms detected
crs_reduced = crs[
    (crs['stat_keywords'].notna()) |
    (crs['stat_blacklist'].notna()) |
    (crs['gen_keywords'].notna()) |
    (crs['stat_acronyms'].notna()) |
    (crs['gen_acronyms'].notna()) 
]

In [20]:
# To xlsx 
#crs_reduced.to_excel("../../data/processed/crs_titles_matched.xlsx", index=False)
crs_reduced.to_excel("../../data/processed/crs_titles_matched_wo_stopwords.xlsx", index=False)

## Make final makers 

In [46]:
# Load raw crs data from feather file
crs_raw = pd.read_feather("../../data/raw/crs_raw.feather")

In [47]:
# Reduce crs_raw to only the columns we need
crs_raw = crs_raw[['project_title', 'short_description', 'long_description', 'purpose_code', 'channel_code', 'donor_code', 'agency_code','gender', 'rmnch', 'sd_gfocus']]

In [48]:
# Load the matched CRS data from feather file
crs_matched = pd.read_feather("../../data/processed/crs_titles_matched_wo_stopwords.feather")

# Keep only the columns project_title, stat_keywords, stat_blacklist, gen_keywords, stat_acronyms, gen_acronyms
crs_matched = crs_matched[['project_title', 'language', 'stat_keywords', 'stat_blacklist', 'gen_keywords', 'stat_acronyms', 'gen_acronyms']]

# Rename lanaguage to title_language
crs_matched.rename(columns={'language': 'title_language'}, inplace=True)

In [49]:
# Left join matched CRS data to the the raw CRS data based on the project_title column
crs = pd.merge(crs_raw, crs_matched, on='project_title', how='left')

del crs_raw
del crs_matched

In [50]:
# Convert chennal_code to int, gender to int, rmnch to int
crs['channel_code'] = crs['channel_code'].astype('Int64')
crs['gender'] = crs['gender'].astype('Int64')
crs['rmnch'] = crs['rmnch'].astype('Int64')
crs['agency_code'] = crs['agency_code'].astype('Int64')
crs['donor_code'] = crs['donor_code'].astype('Int64')

#### Make final statistics marker for text mining


In [ ]:
# Make statistics marker as purpose_code == 16062 | stat_keywords is not null | stat_acronyms is not null & stat_blacklist is null & pupose_code != 15250
crs['is_statistics'] = (
    (
        (crs['purpose_code'] == 16062) |    # purpose_code is 16062
        (crs['stat_keywords'].notna()) |    # stat_keywords is not null
        (crs['stat_acronyms'].notna())      # stat_acronyms is not null
    ) &
    (crs['stat_blacklist'].isna()) &    # stat_blacklist must be null
    (crs['purpose_code'] != 15250)      # purpose_code must not be 15250
)

crs['is_mining'] = (
    (crs['purpose_code'] == 15250) |    # purpose_code is 16062
    (crs['stat_blacklist'].notna())     # stat_blacklist is not null
)

# Set is_statistics to False for title_language == it and stat_acronyms == ' ai ' 
crs.loc[
    (crs['title_language'] == 'it') & 
    (crs['stat_acronyms'].str.contains(' ai ', na=False)), 
    'is_statistics'
] = False

crs['is_statistics'] = crs['is_statistics'].fillna(False)  # Fill NaN values with False (inlcuddes those without project_title)
crs['is_mining'] = crs['is_mining'].fillna(False)  # Fill NaN values with False (inlcuddes those without project_title)

#### Make final gender marker 

In [52]:
# Make gen_donor as (donor_code == 1 & agency_code == 16) | (donor_code == 5 & agency_code == 65)
crs['gen_donor'] = (
    ((crs['donor_code'] == 1) & (crs['agency_code'] == 16)) |  # donor_code is 1 and agency_code is 16
    ((crs['donor_code'] == 5) & (crs['agency_code'] == 65))   # donor_code is 5 and agency_code is 65
)

# Set gen_donor to False if gen_donor is null
crs['gen_donor'] = crs['gen_donor'].fillna(False)

# Make gender marker as:
# - purpose_code is in the range 15170 to 15180 (inclusive)
# - OR gender equals 2
# - OR gen_keywords is not null
# - OR gen_acronyms is not null
# - OR rmnch equals 2
# - OR channel_code is in [41146, 21053, 21040, 21037, 21010]
# - OR gen_donor is True
crs['is_gender'] = (
    (crs['purpose_code'].between(15170, 15180)) |  # purpose_code is in 15170:15180
    (crs['gender'] == 2) |  # gender is 2
    (crs['gen_keywords'].notna()) |  # gen_keywords is not null
    (crs['gen_acronyms'].notna()) |  # gen_acronyms is not null
    (crs['rmnch'] == 2) |  # rmnch is 2
    (crs['channel_code'].isin([41146, 21053, 21040, 21037, 21010])) |  # channel_code 
    (crs['gen_donor'])  # gen_donor is True
) 

# Set is_gender to False if is_gender is null   
crs['is_gender'] = crs['is_gender'].fillna(False)

#### Reduce CRS data with markers to descriptions for text mining 

In [53]:
# Replace missing values or "" of long_descriptions with "N/A"
crs['long_description'] = crs['long_description'].replace("", "N/A")
crs['long_description'] = crs['long_description'].fillna("N/A")

# Create text mining descripton: join short_description, ": ", long_description 
crs['text_mining_description'] = crs['short_description'].astype(str) + ": " + crs['long_description'].astype(str)

# Create text mining descripton: if long_description is not null, use it, else use short_description
#crs['text_mining_description'] = crs['long_description'].combine_first(crs['short_description'])

In [10]:
# Add language column to crs using detect_language function
from src.text_processing import detect_language
crs['language'] = crs['text_mining_description'].apply(detect_language)

# Set all languages that are not in ['en', 'fr', 'es', 'de', 'nl', 'pt'] to 'en'
crs['language'] = crs['language'].apply(lambda x: x if x in ['en', 'fr', 'es', 'de', 'nl', 'pt', 'it'] else 'en')

In [ ]:
# Keep only text_mining_description and gender & statistics markers
crs = crs[['text_mining_description', 'language', 'is_statistics', 'is_mining', 'is_gender']]

In [13]:
# Group by 'text_mining_description' and check if 'is_statistics' has more than one unique value
duplicated_stats = crs.groupby('text_mining_description')['is_statistics'].nunique()

# Filter for descriptions with more than one unique value in 'is_statistics'
conflicting_stats = duplicated_stats[duplicated_stats > 1].index.tolist()

# Create a DataFrame with only the rows that match the duplicated descriptions
conflicting_descr_stat = crs[crs['text_mining_description'].isin(conflicting_stats)]['text_mining_description'].unique()

# Save conflicting_descr_stat to feather file to predict during text mining
conflicting_descr_stat_df = pd.DataFrame(conflicting_descr_stat, columns=['text_mining_description'])
#conflicting_descr_stat_df.to_feather("../../data/processed/conflicting_descr_stat.feather")

In [ ]:
# Group by 'text_mining_description' and check if 'is_gender' has more than one unique value
duplicated_gen = crs.groupby('text_mining_description')['is_gender'].nunique()

# Filter for descriptions with more than one unique value in 'is_gender'
conflicting_gen = duplicated_gen[duplicated_gen > 1].index

# Create a DataFrame with only the rows that match the duplicated descriptions
conflicting_descr_gen = crs[crs['text_mining_description'].isin(conflicting_gen)]['text_mining_description'].unique()

# Save conflicting_descr_gen to feather file to predict during text mining
conflicting_descr_gen_df = pd.DataFrame(conflicting_descr_gen, columns=['text_mining_description'])
conflicting_descr_gen_df.to_feather("../../data/processed/conflicting_descr_gen.feather")

In [14]:
# Discard rows which have conlficting is_statistics value (True & False for the same description)
stat_mining_set = crs[~crs['text_mining_description'].isin(conflicting_descr_stat)].copy()

# Drop duplicates in the stat_mining_set
stat_mining_set = stat_mining_set.drop_duplicates(subset=['text_mining_description'], keep='first')

stat_mining_set = stat_mining_set[['text_mining_description', 'language', 'is_statistics', 'is_mining']].reset_index(drop=True)

In [ ]:
# Discard rows which have conlficting is_stat value (True & False for the same description)
gen_mining_set = crs[~crs['text_mining_description'].isin(conflicting_descr_gen)].copy()

# Drop duplicates in the gen_mining_set
gen_mining_set = gen_mining_set.drop_duplicates(subset=['text_mining_description'], keep='first')

gen_mining_set = gen_mining_set[['text_mining_description', 'language', 'is_gender']].reset_index(drop=True)

In [ ]:
# Save to feather
stat_mining_set.to_feather("../../data/processed/stat_to_mine.feather")
gen_mining_set.to_feather("../../data/processed/gen_to_mine.feather")

# Save only is_statistics projects from stat_to_mine to xlsx 
stat_mining_set[stat_mining_set['is_statistics'] == True].to_excel("../../data/processed/stat_to_mine.xlsx", index=False)

# Save only is_gender projects from gen_to_mine to xlsx
gen_mining_set[gen_mining_set['is_gender'] == True].to_excel("../../data/processed/gen_to_mine.xlsx", index=False)

## Load and Merge the predicted unlabled to the original CRS data

In [1]:
import pandas as pd

# Load the CRS data from feather file
crs = pd.read_feather("../../data/raw/crs_raw.feather")

In [2]:
import pandas as pd

# Read stat_mining_set from feather file
stat_mining_set = pd.read_feather("../../data/processed/stat_to_mine.feather")

# Load the unlabled_predicted data from feather file
unlabeled_predicted = pd.read_feather("../../data/processed/unlabeled_predicted_large_v3.feather")

# Load conflicting_descr_stat_predicted from feather file
conflicting_stat_predicted = pd.read_feather("../../data/processed/conflicting_stat_predicted_large.feather")

In [4]:
unlabeled_predicted[unlabeled_predicted['probability_is_statistics'] > 0.99].to_excel(
    "../../data/processed/unlabeled_predicted_large.xlsx", index=False
)

In [3]:
# Load the matched CRS data from feather file
crs_title_matched = pd.read_feather("../../data/processed/crs_titles_matched_wo_stopwords.feather")

In [4]:
# Load the matched CRS data from feather file
crs_title_matched = pd.read_feather("../../data/processed/crs_titles_matched_wo_stopwords.feather")

# Keep only the columns project_title, stat_keywords, stat_blacklist, gen_keywords, stat_acronyms, gen_acronyms
crs_title_matched = crs_title_matched[['project_title', 'language', 'stat_keywords', 'stat_blacklist', 'gen_keywords', 'stat_acronyms', 'gen_acronyms']]

# Rename lanaguage to title_language
crs_title_matched.rename(columns={'language': 'title_language'}, inplace=True)

# Left join matched CRS data to the the raw CRS data based on the project_title column
crs = pd.merge(crs, crs_title_matched, on='project_title', how='left')

del crs_title_matched

In [5]:
# Merge the stat_mining_set with the unlabeled_predicted data on 'text_mining_description'
stat_mining_predicted = pd.merge(
    stat_mining_set,
    unlabeled_predicted[['text_mining_description', 'probability_is_statistics']],
    on='text_mining_description',
    how='left'
)

del stat_mining_set, unlabeled_predicted

# If is_statistics is True, set probability_is_statistics to 1
stat_mining_predicted.loc[stat_mining_predicted['is_statistics'] == True, 'probability_is_statistics'] = 1

# If is_mining is True, set probability_is_statistics to 0
stat_mining_predicted.loc[stat_mining_predicted['is_mining'] == True, 'probability_is_statistics'] = 0

In [6]:
# Only keep text_mining_description and probability_is_statistics columns
stat_mining_predicted = stat_mining_predicted[['text_mining_description', 'probability_is_statistics']]

# Concanate rows of conflicting_descr_stat_predicted to stat_mining_predicted
stat_mining_predicted = pd.concat([stat_mining_predicted, conflicting_stat_predicted], ignore_index=True)

In [7]:
# Replace missing values or "" of long_descriptions with "N/A"
crs['long_description_for_mining'] = crs['long_description'].replace("", "N/A")
crs['long_description_for_mining'] = crs['long_description_for_mining'].fillna("N/A")

# Create text mining descripton: join short_description, ": ", long_description 
crs['text_mining_description'] = crs['short_description'].astype(str) + ": " + crs['long_description_for_mining'].astype(str)

# Drop long_description_for_mining column
crs.drop(columns=['long_description_for_mining'], inplace=True)

In [8]:
# Merge the stat_mining_predicted with the CRS data on 'text_mining_description'
crs = pd.merge(
    crs,
    stat_mining_predicted[['text_mining_description', 'probability_is_statistics']],
    on='text_mining_description',
    how='left'
)

del stat_mining_predicted

In [9]:
# Overwrite the probability_is_statistics from original title matches for projects in conflicting_descr_stat that had a None title and statistics title with the same text_minig_description 
# IF purpose_code == 16062 | stat_keywords is not null | stat_acronyms is not null & stat_blacklist is null & pupose_code != 15250, set probability_is_statistics to 1
crs.loc[
    (
        (crs['purpose_code'] == 16062) |    # purpose_code is 16062
        (crs['stat_keywords'].notna()) |    # stat_keywords is not null
        (crs['stat_acronyms'].notna())      # stat_acronyms is not null
    ) &
    (crs['stat_blacklist'].isna()) &    # stat_blacklist must be null
    (crs['purpose_code'] != 15250),      # purpose_code must not be 15250 
    'probability_is_statistics'
] = 1

crs.loc[
    (crs['purpose_code'] == 15250) |    # purpose_code is 16062
    (crs['stat_blacklist'].notna()),      # purpose_code must not be 15250 
    'probability_is_statistics'
] = 0

In [10]:
# How many rows have NaN in probability_is_statistics
crs['probability_is_statistics'].isna().sum()

0

In [11]:
# Save as crs_predicted.feather
crs.to_feather("../../data/output/crs_predicted.feather")

## Create Final PRESS from Prediction Results

In [ ]:
# Load the predicted CRS data from feather file
crs = pd.read_feather("../../data/output/crs_predicted.feather")

In [12]:
# Print all column names in crs 
crs.columns.tolist()

['year',
 'donor_code',
 'de_donorcode',
 'donor_name',
 'agency_code',
 'agency_name',
 'crs_id',
 'project_number',
 'initial_report',
 'recipient_code',
 'de_recipientcode',
 'recipient_name',
 'region_code',
 'de_regioncode',
 'region_name',
 'incomegroup_code',
 'de_incomegroup_code',
 'incomegroup_name',
 'flow_code',
 'flow_name',
 'bi_multi',
 'category',
 'finance_t',
 'aid_t',
 'usd_commitment',
 'usd_disbursement',
 'usd_received',
 'usd_commitment_defl',
 'usd_disbursement_defl',
 'usd_received_defl',
 'usd_adjustment',
 'usd_adjustment_defl',
 'usd_amount_untied',
 'usd_amount_partial_tied',
 'usd_amount_tied',
 'usd_amount_untied_defl',
 'usd_amount_partial_tied_defl',
 'usd_amounttied_defl',
 'usd_irtc',
 'usd_expert_commitment',
 'usd_expert_extended',
 'usd_export_credit',
 'currency_code',
 'commitment_national',
 'disbursement_national',
 'grant_equiv',
 'usd_grant_equiv',
 'short_description',
 'project_title',
 'purpose_code',
 'purpose_name',
 'sector_code',
 'sec

In [ ]:
# Filter CRS for projects with probability_is_statistics >= 0.9996
press_2025 = crs[crs['probability_is_statistics'] >= 0.9996].copy()

In [52]:
# Remove projects manually after indetification of errors: 
#  - lowercased project_title conatains international accountability
#  - stat_acronyms == ' nsa '
#  - stat_acronyms == ' ai ' & title_language == 'it'
#  - stat_acronyms == ' ki '
#  - stat_acronyms == ' odw '
import numpy as np

# Exclude rows where project_title contains 'international accountability' and 'national account' is in stat_keywords
press_2025 = press_2025[~(
    press_2025['project_title'].str.lower().str.contains('national accountability', na=False) &
    press_2025['stat_keywords'].apply(
        lambda x: any(kw == 'national account' for kw in x) if isinstance(x, (list, np.ndarray)) else False
    )
)]

# Exclude rows where 'nsa' is in stat_acronyms
press_2025 = press_2025[~press_2025['stat_acronyms'].apply(
    lambda x: any(acronym  == ' nsa ' for acronym in x) if isinstance(x, (list, np.ndarray)) else False
)]

# Exclude rows where 'ai' is in stat_acronyms and title_language is 'it'
press_2025 = press_2025[~(
    press_2025['stat_acronyms'].apply(
        lambda x: any(acronym == ' ai ' for acronym in x) if isinstance(x, (list, np.ndarray)) else False
    ) & (press_2025['title_language'] == 'it')
)]

# Exclude rows where 'odw' is in stat_acronyms
press_2025 = press_2025[~press_2025['stat_acronyms'].apply(
    lambda x: any(acronym == ' odw ' for acronym in x) if isinstance(x, (list, np.ndarray)) else False
)]

press_2025 = press_2025[~press_2025['stat_acronyms'].apply(
    lambda x: any(acronym == ' ki ' for acronym in x) if isinstance(x, (list, np.ndarray)) else False
)]

#### Add RegionName as in PRESS 2024

In [ ]:
# Load PRESS 2024
press_2024 = pd.read_excel("../../data/auxiliary/PRESS_2024.xlsx")      

# Get RegionName and dac_regionname from press_2024
region_mapping = press_2024[['RegionName', 'recipientcode']].drop_duplicates(keep='first')
region_mapping_for_missing = press_2024[['RegionName', 'dac_regionname']].drop_duplicates(keep='first')

# Rename dac_regionname to regionname
region_mapping_for_missing.rename(columns={'dac_regionname': 'region_name'}, inplace=True)

# Remove row with RegionName == 'Eastern Europe' and regionname == 'South & Central Asia'
region_mapping_for_missing = region_mapping_for_missing[
    (region_mapping_for_missing['RegionName'] != 'Eastern Europe') & 
    (region_mapping_for_missing['region_name'] != 'South & Central Asia')
]

# Rename recipientcode to recipient_code
region_mapping.rename(columns={'recipientcode': 'recipient_code'}, inplace=True)

# Left join region_mapping to press_2025 on region_name
press_2025 = pd.merge(press_2025, region_mapping, on='recipient_code', how='left')

# If RegionName is missing, use the regionname from region_mapping_for_missing by merging base on region_name
press_2025 = pd.merge(press_2025, region_mapping_for_missing, on='region_name', how='left', suffixes=('', '_missing'))

# Replace missing RegionName with region_name_missing
press_2025['RegionName'] = press_2025['RegionName'].fillna(press_2025['RegionName_missing'])

# Drop region_name_missing column
press_2025.drop(columns=['RegionName_missing'], inplace=True)

# Rename region_name to dac_regionname
press_2025.rename(columns={'region_name': 'dac_regionname'}, inplace=True)

# Replace missing RegionName with region_name_missing
press_2025['RegionName'] = press_2025['RegionName'].fillna(press_2025['dac_regionname'])

#### Add ReporteName as in PRESS 2024

In [55]:
# Get ReporterName and donorname from press_2024
reporter_mapping = press_2024[['ReporterName', 'donorname']].drop_duplicates(keep='first')

# Rename donorname to donor_name
reporter_mapping.rename(columns={'donorname': 'donor_name'}, inplace=True)

# Left join reporter_mapping to press_2025 on donorname
press_2025 = pd.merge(press_2025, reporter_mapping, on='donor_name', how='left')

# If ReporterName is NaN (no projects previously), set it to donor_name
press_2025['ReporterName'] = press_2025['ReporterName'].fillna(press_2025['donor_name'])

#### Add DonorType (DAC, non-DAC, multilateral, private)

In [ ]:
# DAC donor types 
donor_categories = pd.read_excel("../../data/auxiliary/donor_categories.xlsx")

# Remove Data Explorer code and DonorNameE from donor_categories
donor_categories = donor_categories[['DonorCode', 'DonorType']]

# Rename DonorCode to donor_code
donor_categories.rename(columns={'DonorCode': 'donor_code'}, inplace=True)

# Left join donor_categories to press_2025 on donor_code
press_2025 = pd.merge(press_2025, donor_categories, on='donor_code', how='left')

#### Add Recipient ISO codes & SIDS classfication as of 2024 (from CRS codebook)

In [ ]:
# Read ISO codes from 'Recipient' sheet 
iso_codes = pd.read_excel("../../data/auxiliary/DAC-CRS-CODES.xlsx", sheet_name="Recipient")

# Add ISO codes 
iso_codes = iso_codes[['Recipient code', 'ISOcode', 'SIDS']]

# Drop rows with recipient_code >= 889
iso_codes = iso_codes[iso_codes['Recipient code'] < 889]

# Rename Recipient Code to recipient_code, ISOcode to iso 
iso_codes.rename(columns={'Recipient code': 'recipient_code', 'ISOcode': 'recipient_iso'}, inplace=True)

# Left join iso_codes to press_2025 on recipient_code
press_2025 = pd.merge(press_2025, iso_codes, on='recipient_code', how='left')

# Set SIDS to True if SIDS 1, False if 0 with as.type(bool)
press_2025['SIDS'] = (press_2025['SIDS'] == 1).astype(bool)

# If recipient_iso is NA, set SIDS to NA
press_2025.loc[press_2025['recipient_iso'].isna(), 'SIDS'] = pd.NA

In [ ]:
# Manually correct recipient_iso codes for some countries that are in the CRS but not in the ISO code list
country_iso_data = [
    {'recipient_name': 'Saudi Arabia', 'recipient_iso': 'SAU'},
    {'recipient_name': 'Croatia', 'recipient_iso': 'HRV'},
    {'recipient_name': 'Anguilla', 'recipient_iso': 'AIA'},
    {'recipient_name': 'Barbados', 'recipient_iso': 'BRB'},
    {'recipient_name': 'Antigua and Barbuda', 'recipient_iso': 'ATG'},
    {'recipient_name': 'Chile', 'recipient_iso': 'CHL'},
    {'recipient_name': 'Cyprus', 'recipient_iso': 'CYP'},
    {'recipient_name': 'Malta', 'recipient_iso': 'MLT'},
    {'recipient_name': 'East African Community', 'recipient_iso': None},
    {'recipient_name': 'Seychelles', 'recipient_iso': 'SYC'},
    {'recipient_name': 'Netherlands Antilles', 'recipient_iso': 'ANT'},
    {'recipient_name': 'Aruba', 'recipient_iso': 'ABW'},
    {'recipient_name': 'States Ex-Yugoslavia unspecified', 'recipient_iso': None},
    {'recipient_name': 'Uruguay', 'recipient_iso': 'URY'},
    {'recipient_name': 'Turks and Caicos Islands', 'recipient_iso': 'TCA'},
    {'recipient_name': 'Gibraltar', 'recipient_iso': 'GIB'},
    {'recipient_name': 'Korea', 'recipient_iso': 'KOR'},
    {'recipient_name': 'Slovenia', 'recipient_iso': 'SVN'},
    {'recipient_name': 'Trinidad and Tobago', 'recipient_iso': 'TTO'},
    {'recipient_name': 'Oman', 'recipient_iso': 'OMN'},
    {'recipient_name': 'Cook Islands', 'recipient_iso': 'COK'},
    {'recipient_name': 'Mayotte', 'recipient_iso': 'MYT'},
]

country_iso_df = pd.DataFrame(country_iso_data)

# Replace missing recipient_iso in press_2025 with country_iso_df
for index, row in country_iso_df.iterrows():
    press_2025.loc[press_2025['recipient_name'] == row['recipient_name'], 'recipient_iso'] = row['recipient_iso']

# Change SIDS status for some countries with manually corrected ISO codes: BRB, ATG, AIA, SYC, ABW, TCA, TTO, COK
press_2025.loc[press_2025['recipient_iso'] == 'BRB', 'SIDS'] = True
press_2025.loc[press_2025['recipient_iso'] == 'ATG', 'SIDS'] = True
press_2025.loc[press_2025['recipient_iso'] == 'AIA', 'SIDS'] = True
press_2025.loc[press_2025['recipient_iso'] == 'SYC', 'SIDS'] = True
press_2025.loc[press_2025['recipient_iso'] == 'ABW', 'SIDS'] = True
press_2025.loc[press_2025['recipient_iso'] == 'TCA', 'SIDS'] = True
press_2025.loc[press_2025['recipient_iso'] == 'TTO', 'SIDS'] = True
press_2025.loc[press_2025['recipient_iso'] == 'COK', 'SIDS'] = True


#### Add fragile state classification from WB (https://thedocs.worldbank.org/en/doc/3d4356ac2aee9f0b2db90ae9ce49f639-0090082024/original/FCSList-FY06toFY24.pdf)

In [63]:
# Read fragile states data
fragile_states = pd.read_excel("../../data/auxiliary/fragile_states.xlsx")

fragile_states.drop(columns=['country'], inplace=True) 

# Melt the fragile_states DataFrame to long format
fragile_states_long = fragile_states.melt(
    id_vars=['iso'],
    var_name='year',
    value_name='fragile_state'
)

# Convert year to integer and fragile to boolean
fragile_states_long['year'] = fragile_states_long['year'].astype(int)
fragile_states_long['fragile_state'] = fragile_states_long['fragile_state'].fillna(0).astype(bool)

# Rename iso to recipient_iso
fragile_states_long.rename(columns={'iso': 'recipient_iso'}, inplace=True)

# Left join fragile_states_long to press_2025 on recipient_iso and year
press_2025 = pd.merge(
    press_2025,
    fragile_states_long[['recipient_iso', 'year', 'fragile_state']],
    on=['recipient_iso', 'year'],
    how='left'
)

# For all rows that have a recipeint_iso value that is not in fragile_states_long, set fragile_state to False for values > 2008
press_2025.loc[
    (press_2025['fragile_state'].isna()) & (press_2025['recipient_iso'].notna()) & (press_2025['year'] > 2008),
    'fragile_state'
] = False


In [64]:
press_2025[press_2025['recipient_iso'].isna()]['recipient_name'].unique()

array(['South of Sahara, regional', 'Bilateral, unspecified',
       'Asia, regional', 'Africa, regional', 'South America, regional',
       'South & Central Asia, regional', 'Far East Asia, regional',
       'America, regional', 'Europe, regional', 'Caribbean, regional',
       'Caribbean & Central America, regional', 'Oceania, regional',
       'Central Asia, regional', 'Middle East, regional',
       'South Asia, regional', 'North of Sahara, regional',
       'Western Africa, regional', 'Central America, regional',
       'Eastern Africa, regional', 'Southern Africa, regional',
       'Melanesia, regional', 'Middle Africa, regional',
       'East African Community', 'States Ex-Yugoslavia unspecified'],
      dtype=object)

#### Output Final PRESS dataset & IMF 2023 dataset 

In [ ]:
# Post 2009 
press_2025_post_2009 = press_2025[press_2025['year'] > 2009].copy()

press_2025_post_2009.to_excel("../../data/output/press_2010-2025.xlsx", index=False)

In [ ]:
# Full dataset
press_2025.to_excel("../../data/output/press_1973-2025.xlsx", index=False)

In [146]:
# IMF data: either donor_name == IMF Resilience and Sustainability Trust or channel_name == IMF
imf_data_2023 = press_2025[
    (press_2025['donor_name'] == 'IMF Resilience and Sustainability Trust') | 
    (press_2025['channel_name'] == 'International Monetary Fund (IMF)') 
].copy()

# keep only for year == 2023
imf_data_2023 = imf_data_2023[imf_data_2023['year'] == 2023]

# Remove unecessary columns 
imf_data_2023 = imf_data_2023[[
    'year',
 'donor_name',
 'agency_name',
 'crs_id',
 'project_number',
 'initial_report',
 'recipient_name',
 'region_code',
 'region_name',
 'incomegroup_name',
 'flow_code',
 'flow_name',
 'bi_multi',
 'category',
 'finance_t',
 'aid_t',
 'usd_commitment',
 'usd_disbursement',
 'usd_received',
 'usd_commitment_defl',
 'usd_disbursement_defl',
 'usd_received_defl',
 'usd_adjustment',
 'usd_adjustment_defl',
 'project_title',
 'short_description',
 'long_description',
 'purpose_code',
 'purpose_name',
 'sector_code',
 'sector_name',
 'channel_code',
 'channel_name',
 'channel_reported_name',
 'parent_channel_code',
 'geography',
 'ld_cflag',
 'ld_cflag_name',
 'expected_start_date',
 'completion_date'
]]

# Save IMF data to excel
imf_data_2023.to_excel("../../data/output/imf_data_2023.xlsx", index=False)